# BLaIR Encoding for Amazon Electronics 2023

Sử dụng `hyp1231/blair-roberta-base` encode lại semantic embedding:

- `gold_item_blair_embeddings.npy`
- `gold_user_blair_embeddings.npy`

Thiết kế chính:

- Chạy GPU CUDA trên Colab.
- Encode theo part/chunk để tránh giữ toàn bộ embedding trong RAM.
- Có resume: chạy lại sẽ tự skip part đã lưu đúng shape.
- Merge bằng `numpy.memmap`, không load full item/user embedding vào RAM.
- Verify bằng `mmap_mode="r"`.

>BLaIR-RoBERTa-base cho embedding **768 chiều**, khi đưa sang training cần set `LLM_DIM = 768` hoặc đọc tự động bằng `item_emb.shape[1]`.


In [ ]:
# 1. Install dependencies

!pip install -q transformers pandas pyarrow numpy tqdm huggingface_hub safetensors

In [ ]:
# 2. Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Config

from pathlib import Path

# Hugging Face public dataset repos
HF_SILVER_REPO = "chuongdo1104/amazon-2023-silver"
HF_GOLD_REPO = "chuongdo1104/amazon-2023-gold"

# Local Colab cache
LOCAL_DATA_ROOT = Path("/content/recsys_data")
LOCAL_SILVER_ROOT = LOCAL_DATA_ROOT / "hf_silver"
LOCAL_GOLD_ROOT = LOCAL_DATA_ROOT / "hf_gold"

SILVER_DIR = LOCAL_SILVER_ROOT / "silver"
GOLD_DIR = LOCAL_GOLD_ROOT / "gold"

# Output nên lưu vào Drive để không mất khi Colab bị ngắt
DRIVE_OUT = Path("/content/drive/MyDrive/new_gold_embeddings_blair")
ITEM_PART_DIR = DRIVE_OUT / "item_parts"
USER_PART_DIR = DRIVE_OUT / "user_parts"

DRIVE_OUT.mkdir(parents=True, exist_ok=True)
ITEM_PART_DIR.mkdir(parents=True, exist_ok=True)
USER_PART_DIR.mkdir(parents=True, exist_ok=True)

# Model
MODEL_NAME = "hyp1231/blair-roberta-base"

ITEM_MAX_LENGTH = 256      #đổi 384 nếu GPU/RAM đủ.
USER_MAX_LENGTH = 256

ITEM_BATCH_SIZE = 64
USER_BATCH_SIZE = 64

# Mỗi part 50k rows:
# 50k x 768 x float32 ≈ 153 MB / part
PART_ROWS = 50_000

# float32 cho downstream ổn định.
# Nếu muốn tiết kiệm dung lượng, có thể thử "float16", nhưng training adapter nên dùng float32.
OUTPUT_DTYPE = "float32"

# Pooling:
# "cls": lấy token đầu tiên <s>.
# "mean": mean pooling theo attention mask.
POOLING = "cls"

RUN_ITEMS = True
RUN_USERS = True

MERGE_AFTER_ENCODE = True

FORCE_REMERGE = False

CLEAN_PARTS_AFTER_SUCCESS = False

print("SILVER_DIR:", SILVER_DIR)
print("GOLD_DIR:", GOLD_DIR)
print("DRIVE_OUT:", DRIVE_OUT)

In [ ]:
# 4. Imports + environment

import os
import json
import gc
import shutil
import time
from typing import Tuple

# Nên set trước khi dùng torch nhiều
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import snapshot_download
from numpy.lib.format import open_memmap

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("VRAM total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("WARNING: Colab chưa bật GPU. Vào Runtime → Change runtime type → GPU.")

In [ ]:
# 5. Download Silver/Gold from Hugging Face

def download_hf_data():
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

    print("\nDownloading SILVER repo...")
    snapshot_download(
        repo_id=HF_SILVER_REPO,
        repo_type="dataset",
        local_dir=str(LOCAL_SILVER_ROOT),
        # Chỉ lấy folder silver để nhẹ hơn
        allow_patterns=["silver/**"],
    )

    print("\nDownloading GOLD repo...")
    snapshot_download(
        repo_id=HF_GOLD_REPO,
        repo_type="dataset",
        local_dir=str(LOCAL_GOLD_ROOT),
        # Gold là flat files trong gold/
        allow_patterns=["gold/**"],
    )

    print("\nDone download.")
    print("Silver exists:", SILVER_DIR.exists(), SILVER_DIR)
    print("Gold exists:", GOLD_DIR.exists(), GOLD_DIR)

download_hf_data()

In [ ]:
# 6. Check required files

def check_required_paths():
    required = {
        "silver_item_text_profile": SILVER_DIR / "silver_item_text_profile.parquet",
        "silver_user_text_profile": SILVER_DIR / "silver_user_text_profile.parquet",
        "gold_item_id_map": GOLD_DIR / "gold_item_id_map.parquet",
        "gold_user_id_map": GOLD_DIR / "gold_user_id_map.parquet",
    }

    print("\nChecking required paths:")
    for name, path in required.items():
        ok = path.exists()
        print(f"  {name}: {path} | exists={ok}")
        if not ok:
            raise FileNotFoundError(f"Missing required path: {path}")

    print("\nPreview local files:")
    !find /content/recsys_data -maxdepth 4 -type f | head -30

check_required_paths()

In [ ]:
# 7. Load tokenizer/model

print("Loading tokenizer:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model:", MODEL_NAME)
if DEVICE == "cuda":
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
    ).to(DEVICE)
else:
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

model.eval()
for p in model.parameters():
    p.requires_grad = False

EMB_DIM = int(model.config.hidden_size)
print("Embedding dim:", EMB_DIM)

assert EMB_DIM == 768, f"Expected BLaIR/RoBERTa-base dim 768, got {EMB_DIM}."


In [ ]:
# 8. Utility functions

def valid_npy(path: Path, expected_shape: Tuple[int, int], expected_dtype: str = OUTPUT_DTYPE) -> bool:
    if not path.exists():
        return False
    try:
        arr = np.load(path, mmap_mode="r")
        ok = (arr.shape == expected_shape) and (str(arr.dtype) == expected_dtype)
        del arr
        return ok
    except Exception:
        return False


def part_path(part_dir: Path, prefix: str, start: int, end: int) -> Path:
    return part_dir / f"{prefix}_{start:09d}_{end:09d}.npy"


def meta_path(npy_path: Path) -> Path:
    return npy_path.with_suffix(".json")


def pool_hidden_state(outputs, attention_mask):
    last_hidden = outputs.last_hidden_state

    if POOLING == "cls":
        # RoBERTa/BLaIR: token đầu tiên là <s>
        emb = last_hidden[:, 0]
    elif POOLING == "mean":
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        summed = (last_hidden * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        emb = summed / denom
    else:
        raise ValueError(f"Unsupported POOLING: {POOLING}")

    return F.normalize(emb.float(), p=2, dim=1)


@torch.inference_mode()
def encode_batch(batch_texts, max_length: int):
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(DEVICE == "cuda"),
    ):
        outputs = model(**inputs, return_dict=True)
        emb = pool_hidden_state(outputs, inputs["attention_mask"])

    vec = emb.cpu().numpy().astype(OUTPUT_DTYPE)

    del inputs, outputs, emb
    return vec


def save_json(path: Path, payload: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def remove_tmp_if_exists(path: Path):
    if path.exists():
        path.unlink()


def cleanup_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


In [ ]:
# 9. Load dataframe functions

def load_item_df() -> pd.DataFrame:
    print("\nLoading item text profile...")
    item_text = pd.read_parquet(
        SILVER_DIR / "silver_item_text_profile.parquet",
        columns=["parent_asin", "item_text"],
    )

    print("Loading item id map...")
    item_map = pd.read_parquet(
        GOLD_DIR / "gold_item_id_map.parquet",
        columns=["parent_asin", "item_idx"],
    )

    print("Merging item_text with item_idx...")
    df = item_map.merge(item_text, on="parent_asin", how="left")

    df["item_text"] = df["item_text"].fillna("")
    df = df.sort_values("item_idx").reset_index(drop=True)

    idx = df["item_idx"].to_numpy()
    assert (idx == np.arange(len(df))).all(), "item_idx không liên tục hoặc sort sai."

    df = df[["item_idx", "item_text"]]
    print("Item df:", df.shape)
    print(df.head(2))
    return df


def load_user_df() -> pd.DataFrame:
    print("\nLoading user text profile...")
    user_text = pd.read_parquet(
        SILVER_DIR / "silver_user_text_profile.parquet",
        columns=["reviewer_id", "user_text"],
    )

    print("Loading user id map...")
    user_map = pd.read_parquet(
        GOLD_DIR / "gold_user_id_map.parquet",
        columns=["reviewer_id", "user_idx"],
    )

    print("Merging user_text with user_idx...")
    df = user_map.merge(user_text, on="reviewer_id", how="left")

    df["user_text"] = df["user_text"].fillna("")
    df = df.sort_values("user_idx").reset_index(drop=True)

    idx = df["user_idx"].to_numpy()
    assert (idx == np.arange(len(df))).all(), "user_idx không liên tục hoặc sort sai."

    df = df[["user_idx", "user_text"]]
    print("User df:", df.shape)
    print(df.head(2))
    return df


In [ ]:
# 10. Encode to parts with resume

def encode_df_to_parts(
    df: pd.DataFrame,
    text_col: str,
    idx_col: str,
    prefix: str,
    part_dir: Path,
    batch_size: int,
    max_length: int,
):
    n = len(df)
    part_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n===== Encoding {prefix} =====")
    print("Rows:", f"{n:,}")
    print("Batch size:", batch_size)
    print("Max length:", max_length)
    print("Part rows:", PART_ROWS)
    print("Embedding dim:", EMB_DIM)
    print("Output dtype:", OUTPUT_DTYPE)
    print("Pooling:", POOLING)

    for start in range(0, n, PART_ROWS):
        end = min(start + PART_ROWS, n)
        out_path = part_path(part_dir, prefix, start, end)
        expected_shape = (end - start, EMB_DIM)

        if valid_npy(out_path, expected_shape):
            print(f"[SKIP] {out_path.name}")
            continue

        print(f"\n[ENCODE] {prefix}: rows {start:,} → {end:,}")

        tmp_path = out_path.with_name(out_path.stem + ".tmp.npy")
        remove_tmp_if_exists(tmp_path)

        # Tạo part file dạng memmap để không cần giữ toàn bộ part trong RAM.
        part_arr = open_memmap(
            tmp_path,
            mode="w+",
            dtype=OUTPUT_DTYPE,
            shape=expected_shape,
        )

        part_df = df.iloc[start:end]
        idxs = part_df[idx_col].to_numpy()
        expected_idxs = np.arange(start, end)

        # Vì df đã sort theo index liên tục, dòng thứ k phải tương ứng idx k.
        assert (idxs == expected_idxs).all(), (
            f"{idx_col} không khớp với thứ tự dòng ở part {start}-{end}"
        )

        texts = part_df[text_col].fillna("").astype(str).tolist()

        for local_start in tqdm(
            range(0, len(texts), batch_size),
            desc=f"{prefix} {start}-{end}",
            leave=True,
        ):
            local_end = min(local_start + batch_size, len(texts))
            batch_texts = texts[local_start:local_end]

            vec = encode_batch(batch_texts, max_length=max_length)

            part_arr[local_start:local_end] = vec
            part_arr.flush()

            del vec, batch_texts

        del part_arr, texts, part_df
        cleanup_gpu()

        # Atomic-ish rename: chỉ sau khi part hoàn tất mới đổi sang tên chính.
        os.replace(tmp_path, out_path)

        save_json(
            meta_path(out_path),
            {
                "prefix": prefix,
                "start": start,
                "end": end,
                "shape": list(expected_shape),
                "dtype": OUTPUT_DTYPE,
                "model": MODEL_NAME,
                "pooling": POOLING,
                "max_length": max_length,
                "batch_size": batch_size,
                "part_rows": PART_ROWS,
                "created_time": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
        )

        assert valid_npy(out_path, expected_shape), f"Saved part invalid: {out_path}"

        print("[SAVED]", out_path.name, "| shape:", expected_shape)

    print(f"\n[DONE PARTS] {prefix}")


In [ ]:
# 11. Merge parts to final .npy with memmap
def merge_parts_to_final(
    part_dir: Path,
    prefix: str,
    final_path: Path,
    num_rows: int,
):
    expected_shape = (num_rows, EMB_DIM)

    if final_path.exists() and not FORCE_REMERGE:
        if valid_npy(final_path, expected_shape):
            print(f"\n[SKIP MERGE] Final exists and valid: {final_path}")
            return final_path
        else:
            print(f"\n[REMERGE] Final exists but invalid: {final_path}")
            final_path.unlink()

    print(f"\n===== Merging {prefix} =====")
    print("Final path:", final_path)
    print("Final shape:", expected_shape)

    tmp_final = final_path.with_name(final_path.stem + ".tmp.npy")
    remove_tmp_if_exists(tmp_final)

    final_arr = open_memmap(
        tmp_final,
        mode="w+",
        dtype=OUTPUT_DTYPE,
        shape=expected_shape,
    )

    for start in tqdm(range(0, num_rows, PART_ROWS), desc=f"Merging {prefix}"):
        end = min(start + PART_ROWS, num_rows)
        p = part_path(part_dir, prefix, start, end)
        part_shape = (end - start, EMB_DIM)

        if not valid_npy(p, part_shape):
            raise FileNotFoundError(f"Missing or invalid part: {p}")

        part_arr = np.load(p, mmap_mode="r")
        final_arr[start:end] = part_arr
        final_arr.flush()

        del part_arr

    del final_arr
    os.replace(tmp_final, final_path)

    assert valid_npy(final_path, expected_shape), f"Final file invalid: {final_path}"

    save_json(
        final_path.with_suffix(".json"),
        {
            "prefix": prefix,
            "shape": list(expected_shape),
            "dtype": OUTPUT_DTYPE,
            "model": MODEL_NAME,
            "pooling": POOLING,
            "embedding_dim": EMB_DIM,
            "part_rows": PART_ROWS,
            "created_time": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
    )

    print("[MERGED]", final_path)
    return final_path


def cleanup_part_dir(part_dir: Path):
    if CLEAN_PARTS_AFTER_SUCCESS:
        print("Cleaning part dir:", part_dir)
        shutil.rmtree(part_dir)


In [ ]:
# 12. Run item encoding

if RUN_ITEMS:
    item_df = load_item_df()

    encode_df_to_parts(
        df=item_df,
        text_col="item_text",
        idx_col="item_idx",
        prefix="gold_item_blair",
        part_dir=ITEM_PART_DIR,
        batch_size=ITEM_BATCH_SIZE,
        max_length=ITEM_MAX_LENGTH,
    )

    if MERGE_AFTER_ENCODE:
        item_final = merge_parts_to_final(
            part_dir=ITEM_PART_DIR,
            prefix="gold_item_blair",
            final_path=DRIVE_OUT / "gold_item_blair_embeddings.npy",
            num_rows=len(item_df),
        )
        cleanup_part_dir(ITEM_PART_DIR)

    del item_df
    cleanup_gpu()
else:
    print("Skip item encoding.")


In [ ]:
# 13. Run user encoding

if RUN_USERS:
    user_df = load_user_df()

    encode_df_to_parts(
        df=user_df,
        text_col="user_text",
        idx_col="user_idx",
        prefix="gold_user_blair",
        part_dir=USER_PART_DIR,
        batch_size=USER_BATCH_SIZE,
        max_length=USER_MAX_LENGTH,
    )

    if MERGE_AFTER_ENCODE:
        user_final = merge_parts_to_final(
            part_dir=USER_PART_DIR,
            prefix="gold_user_blair",
            final_path=DRIVE_OUT / "gold_user_blair_embeddings.npy",
            num_rows=len(user_df),
        )
        cleanup_part_dir(USER_PART_DIR)

    del user_df
    cleanup_gpu()
else:
    print("Skip user encoding.")


In [ ]:
# 14. Verify final files safely with mmap

item_path = DRIVE_OUT / "gold_item_blair_embeddings.npy"
user_path = DRIVE_OUT / "gold_user_blair_embeddings.npy"

if item_path.exists():
    item_emb = np.load(item_path, mmap_mode="r")
    print("Item final:", item_path)
    print("  shape:", item_emb.shape)
    print("  dtype:", item_emb.dtype)
    print("  sample norm[0]:", float(np.linalg.norm(item_emb[0])))
    del item_emb
else:
    print("Item final not found:", item_path)

if user_path.exists():
    user_emb = np.load(user_path, mmap_mode="r")
    print("User final:", user_path)
    print("  shape:", user_emb.shape)
    print("  dtype:", user_emb.dtype)
    print("  sample norm[0]:", float(np.linalg.norm(user_emb[0])))
    del user_emb
else:
    print("User final not found:", user_path)


## Sửa config training sau khi encode

Khi đưa embedding BLaIR vào LightGCN/adaptor, không để `LLM_DIM=384` nữa. Nên đọc tự động:

```python
item_emb_llm = np.load("gold_item_blair_embeddings.npy", mmap_mode="r")
user_emb_llm = np.load("gold_user_blair_embeddings.npy", mmap_mode="r")

assert item_emb_llm.shape[0] == num_items
assert user_emb_llm.shape[0] == num_users
assert item_emb_llm.shape[1] == user_emb_llm.shape[1]

CFG["LLM_DIM"] = item_emb_llm.shape[1]  # BLaIR thường là 768
```

Adapter sau đó sẽ học phép chiếu:

```text
BLaIR text embedding [768] → LightGCN embedding_dim [64/128/...]
```


In [ ]:
# 15. Optional: copy final files from Drive to local SSD for faster training in same Colab

COPY_TO_LOCAL_CACHE = False

if COPY_TO_LOCAL_CACHE:
    LOCAL_CACHE = Path("/content/recsys_cache")
    LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

    for src in [
        DRIVE_OUT / "gold_item_blair_embeddings.npy",
        DRIVE_OUT / "gold_user_blair_embeddings.npy",
    ]:
        if src.exists():
            dst = LOCAL_CACHE / src.name
            print("Copy:", src, "->", dst)
            shutil.copy2(src, dst)

    print("Local cache files:")
    !ls -lh /content/recsys_cache
